# Train a face attack decision model

Fine-tunes `unsloth/Qwen3.5-4B` with Unsloth's `FastDecisionModel` so that, for one cropped face image, it answers:

* `is_bona_fide` (yes/no): is this a live, unaltered face?
* `attack_type` (choice): bona fide, print, replay, mask, face swap, face reenactment or fully synthetic.

Then it scores the model on datasets it never saw, with ISO/IEC 30107-3 metrics.

**Runtime:** Colab Pro with an L4 or A100 GPU. On a T4, Qwen3.5 trains in float32 and is very slow.

**Smoke mode** (`SMOKE = True`, the default) needs no data: it generates noise images to check that the pipeline runs. Only the repository has to be reachable through `REPO_SOURCE`.

**Before a real run:** put the face crops and a `manifest.csv` in Google Drive (see `docs/data.md`), and copy this repository to Drive.

**Licenses:** the public face datasets are research-only, and Unsloth's decision model code is AGPL-3.0. Keep the trained adapter for research.

### Configuration

In [ ]:
# SMOKE = True runs the whole pipeline on generated noise images with arbitrary labels. It checks
# that installation, loading, training, calibration and prediction work on this GPU; its metrics
# mean nothing. Set it to False once the real face crops and manifest are in Drive.
SMOKE = True

DRIVE_ROOT = "/content/drive/MyDrive"
# Where pip installs this repository from: a copy in Drive or a git URL ("git+https://...").
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
OUTPUT_DIR = f"{DRIVE_ROOT}/face-decision-runs/qwen35_4b" + ("_smoke" if SMOKE else "")

if SMOKE:
    DATA_DIR = "/content/smoke_data"
    TRAIN_SOURCES, TEST_SOURCES = {"smoke_train"}, {"smoke_test"}
    MAX_STEPS = 5
    MAX_TEST_SAMPLES = None
else:
    DATA_DIR = f"{DRIVE_ROOT}/face-decision-data"  # face crops; manifest image_path is relative to it
    # Cross-dataset protocol: never test on a dataset used for training.
    TRAIN_SOURCES = {"ffpp", "celeba_spoof"}
    TEST_SOURCES = {"celebdf_v2", "oulu_npu"}
    MAX_STEPS = 60                  # short run; set to -1 for a full epoch
    MAX_TEST_SAMPLES = 4000         # cap for the evaluation loop; None scores every test crop
MANIFEST = f"{DATA_DIR}/manifest.csv"

CALIBRATION_FRACTION = 0.1   # of training subjects, held out to calibrate probabilities
MAX_APCER = 0.05             # operating point reported as BPCER @ APCER <= this
SEED = 3407

### Installation

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%capture
import importlib.util
import os

!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
!uv pip install -qqq "{REPO_SOURCE}"

### Data

Splits come from `face_decision_lab.splits`, not from `FastDecisionModel.split_holdout`: that one shuffles rows, and since every frame is a row, frames of one video would land on both sides.

In [ ]:
from collections import Counter

from face_decision_lab.manifest import load_manifest
from face_decision_lab.rows import decision_row
from face_decision_lab.splits import cross_dataset_split, group_split

if SMOKE:
    from face_decision_lab.smoke import write_smoke_dataset
    write_smoke_dataset(DATA_DIR, seed = SEED)

samples = load_manifest(MANIFEST)
train_samples, test_samples = cross_dataset_split(samples, TRAIN_SOURCES, TEST_SOURCES)
train_samples, calibration_samples = group_split(train_samples, CALIBRATION_FRACTION, seed = SEED)

for name, part in [("train", train_samples), ("calibration", calibration_samples), ("test", test_samples)]:
    print(name, len(part), dict(Counter(s.attack_type for s in part)))

### Model

In [ ]:
from unsloth import DecisionTrainer, FastDecisionModel, is_bfloat16_supported

model, processor = FastDecisionModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 2048,
    load_in_4bit = True,
)
# LoRA trains the language layers; Unsloth keeps the vision tower frozen.
model = FastDecisionModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

In [ ]:
train_items, train_report = FastDecisionModel.build_dataset(
    [decision_row(s, DATA_DIR) for s in train_samples], processor, model
)
calibration_items, calibration_report = FastDecisionModel.build_dataset(
    [decision_row(s, DATA_DIR) for s in calibration_samples], processor, model
)
print("train", train_report)
print("calibration", calibration_report)

### Train

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = processor,
    train_dataset = train_items,
    eval_dataset = calibration_items,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 10,
        output_dir = f"{OUTPUT_DIR}/checkpoints",
        report_to = "none",
        seed = SEED,
    ),
)
trainer_stats = trainer.train()

### Calibrate on held-out training subjects

In [ ]:
FastDecisionModel.calibrate(model, processor, calibration_items)

### Evaluate on unseen datasets

`attack_score` is `1 - P(is_bona_fide = true)`. Metrics follow ISO/IEC 30107-3: APCER is reported for the worst attack type.

In [ ]:
import random

from PIL import Image

from face_decision_lab.labels import QUESTIONS, STATE
from face_decision_lab.metrics import auc, bpcer_at_apcer, eer
from face_decision_lab.rows import attack_score

FastDecisionModel.for_inference(model)
evaluated = list(test_samples)
if MAX_TEST_SAMPLES is not None and len(evaluated) > MAX_TEST_SAMPLES:
    evaluated = random.Random(SEED).sample(evaluated, MAX_TEST_SAMPLES)

scores, types, type_hits = [], [], 0
for sample in evaluated:
    with Image.open(f"{DATA_DIR}/{sample.image_path}") as image:
        answers = FastDecisionModel.predict(model, processor, STATE, QUESTIONS, images = [image.convert("RGB")])
    scores.append(attack_score(answers))
    types.append(sample.attack_type)
    type_hits += answers["attack_type"]["answer"] == sample.attack_type

point = bpcer_at_apcer(scores, types, MAX_APCER)
print(f"samples: {len(scores)}")
print(f"AUC: {auc(scores, types):.4f}   EER: {eer(scores, types):.4f}")
print(f"BPCER @ APCER <= {MAX_APCER}: {point.bpcer:.4f} (threshold {point.threshold:.4f})")
print("APCER by attack type:", {k: round(v, 4) for k, v in point.apcer_by_type.items()})
print(f"attack_type accuracy: {type_hits / len(scores):.4f}")

### Save

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/adapter")  # LoRA adapter, decision head and calibration